In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/moc_results', exist_ok=True)

Mounted at /content/drive


In [2]:
import os

if os.path.exists('/content/dvlm-project'):
    %cd /content/dvlm-project
    !git pull
else:
    !git clone https://github.com/ShumailaJaved/dvlm-project.git /content/dvlm-project
    %cd /content/dvlm-project

print("Working directory:", os.getcwd())
!ls

Cloning into '/content/dvlm-project'...
remote: Enumerating objects: 6404, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 6404 (delta 28), reused 24 (delta 14), pack-reused 6343 (from 2)
Receiving objects: 100% (6404/6404), 469.49 MiB | 19.08 MiB/s, done.
Resolving deltas: 100% (116/116), done.
Updating files: 100% (7920/7920), done.
/content/dvlm-project
Working directory: /content/dvlm-project
checkpoints  eval	 LLaVA	    paper	setup_qlora.py
connector    eval_e1.py  losses     patch3.ps1	train_moc.py
data	     figures	 notebooks  results	train_single.py


In [3]:
!pip install -q "bitsandbytes>=0.41" "peft>=0.9" accelerate "transformers>=4.37" datasets einops tqdm
!pip install -q --upgrade torchao
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.6 MB/s eta 0:00:00
Done.


In [4]:
!pip install -q --no-deps -e ./LLaVA
import torch
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llava (pyproject.toml) ... done
torch: 2.11.0+cu128
CUDA available: True


Restart runtime manually before running the remaining cells.

In [1]:
import os
%cd /content/dvlm-project
print("Working directory:", os.getcwd())

/content/dvlm-project
Working directory: /content/dvlm-project


In [2]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### Run the above 2 cells before any training.

Hugging Face tokens increases download speed for models. You can create a new token by going to https://huggingface.co/settings/tokens after logging-in and then clicking 'Create new token'. Copy it and save to Colab secrets (the button with key symbol on left column). Then 'Add new secret', set token name to HF_TOKEN, enable notebook access and paste the token.

## Full MoC Training Run

In [ ]:
!python train_moc.py \
    --epochs 2 \
    --batch 4 \
    --val_every 750 \
    --n_val 200 \
    --n_test 500 \
    --out_dir results/moc

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so

  Full MoC Training
  Epochs: 2  |  Batch: 4×4=16
  L_total = L_CE + 0.1 * L_lb

Loading LLaVA-1.5-7B in 4-bit NF4 …
config.json: 100% 1.16k/1.16k [00:00<00:00, 2.41MB/s]
tokenizer_config.json: 100% 749/749 [00:00<00:00, 5.22MB/s]
tokenizer.model: 100% 500k/500k [00:01<00:00, 399kB/s]
special_tokens_map.json: 100% 438/438 [00:00<00:00, 3.12MB/s]
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
pytorch_model.bin.index.json: 100% 27.1k/27.1k [00:00<00:00, 66.7MB/s]

model.safete

In [ ]:
import shutil, os

src = '/content/dvlm-project/results/moc'
dst = '/content/drive/MyDrive/moc_results/moc'
os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
print("MoC results saved to Drive.")
!ls /content/drive/MyDrive/moc_results/moc/

MoC results saved to Drive.


## Expert 1 Training

In [ ]:
!python train_single.py \
    --expert E1 \
    --epochs 2 \
    --batch 4 \
    --n_eval 200 \
    --out_dir results/single

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so

  Single-Expert Training: E1
  Epochs: 2  |  Batch: 4×4=16

Loading LLaVA-1.5-7B in 4-bit NF4 …
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Fetching 2 files: 100% 2/2 [00:00<00:00, 35544.95it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights:   1% 4/295 [00:00<00:06, 42.05it/s, Materializing param=model.layers.0.mlp.down_proj.weight]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a f

### training finished correctly for E1 but eval has problems^

In [ ]:
import shutil, os

src = '/content/dvlm-project/results/single'
dst = '/content/drive/MyDrive/moc_results/single'
os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
print("E1 results saved to Drive.")

E1 results saved to Drive.


In [4]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.29 KiB | 1.29 MiB/s, done.
From https://github.com/ShumailaJaved/dvlm-project
   dd49f1d..d50c84d  main              -> origin/main
   b9c84eb..0f5df68  integrate/ckpt-e1 -> origin/integrate/ckpt-e1
Updating dd49f1d..d50c84d
Fast-forward
 eval_e1.py | 7 +++++--
 1 file changed, 5 insertions(+), 2 deletions(-)


## Standalone script for E1 eval

In [4]:
!python eval_e1.py --n_eval 500

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading LLaVA-1.5-7B in 4-bit NF4 …
config.json: 100% 1.16k/1.16k [00:00<00:00, 2.19MB/s]
tokenizer_config.json: 100% 749/749 [00:00<00:00, 3.41MB/s]
tokenizer.model: 100% 500k/500k [00:01<00:00, 395kB/s]
special_tokens_map.json: 100% 438/438 [00:00<00:00, 2.32MB/s]
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
pytorch_model.bin.index.json: 100% 27.1k/27.1k [00:00<00:00, 60.8MB/s]

model.safetensors.index.json: 100% 28.4k/28.4k [00:00<00:00, 66.2MB/s]
Fetching 2 files: 100% 

completed eval for E1 using a standalone script and previous errors have been fixed in both scripts train_single.py and train_moc.py

In [5]:
import shutil, os

src = '/content/dvlm-project/results/single'
dst = '/content/drive/MyDrive/moc_results/single'
os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
print("Saved to Drive.")

Saved to Drive.
